# Scenario 03: MCP Tool Integration

**QE Perspective:** OGX can invoke tools exposed via the Model Context Protocol (MCP). This notebook validates:

- Tool-calling via an MCP server using the `mcp::` prefix in the `tools` list.
- The agent calls a tool (e.g. a dummy tool such as "add" or "get_greeting") and synthesizes the result into a response.

**Prerequisite:** An MCP server must be running and registered with OGX (e.g. a tool group). Use `mcp::<toolgroup_id>` in the tools parameter.

Configuration: `OGX_BASE_URL` / `MODEL_ID` or `BASE_URL` / `MODEL`. Server assumed at `http://localhost:8321`.

## Setup

Load base URL and model from environment; create the OGX client.

In [1]:
import os

base_url = os.environ.get("OGX_BASE_URL") or os.environ.get(
    "BASE_URL", "http://localhost:8321"
)
base_url = base_url.rstrip("/")
if base_url.endswith("/v1"):
    base_url = base_url[:-3].rstrip("/")
model = os.environ.get("MODEL_ID") or os.environ.get("MODEL")
mcp_tool_group = os.environ.get("MCP_TOOL_GROUP", "mcp::my-server")

assert base_url, "OGX_BASE_URL or BASE_URL must be set"
assert model, "MODEL_ID or MODEL must be set"

from ogx_client import OgxClient

client = OgxClient(base_url=base_url)

AssertionError: MODEL_ID or MODEL must be set

## MCP usage: minimal server definition

Define a minimal MCP server with the `mcp` package: one **tool** (`add`) and one **resource** (greeting by name). Run this server (e.g. via `app.run(transport="streamable-http")`) and register it with OGX so the agent can call it.

In [ ]:
from mcp.server.fastmcp import FastMCP

app = FastMCP("my-server")


@app.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@app.resource("greeting://{name}")
def get_greeting(name: str) -> str:
    """Get a personalized greeting."""
    return f"Hello, {name}!"


# To run (e.g. in a separate process): app.run(transport="streamable-http")

## Request with MCP tool

Call the Responses API with `tools=[mcp::...]` so the agent can invoke the MCP tool (e.g. "add" or use the greeting). We assert that the response completes and that the model used the tool (tool_calls present) or that the final message reflects the tool result.

In [ ]:
response = None
_skip_reason = None

try:
    response = client.responses.create(
        model=model,
        input="Use the available tool to add 2 and 3. Tell me the result.",
        tools=[
            {
                "type": "mcp",
                "server_label": mcp_tool_group.replace("mcp::", ""),
                "server_url": "http://localhost:8321",
            }
        ],
        stream=False,
    )
except Exception as e:
    error_msg = str(e)
    if (
        "500" in error_msg
        or "not found" in error_msg.lower()
        or "not registered" in error_msg.lower()
    ):
        _skip_reason = f"MCP server not available, skipping: {error_msg[:150]}"
        print(f"SKIPPED: {_skip_reason}")
    else:
        raise

if response is not None:
    assert response.status == "completed", (
        f"Expected status completed, got {response.status}"
    )
    assert response.output is not None, "Expected output to be present"

    has_tool_usage = False
    full_text = ""
    if response.output:
        for item in response.output:
            item_type = getattr(item, "type", None)
            if item_type == "message" and getattr(item, "content", None):
                for c in item.content:
                    if getattr(c, "text", None):
                        full_text += c.text
            if item_type in ("function_call", "mcp_call") or getattr(
                item, "tool_calls", None
            ):
                has_tool_usage = True
    if getattr(response, "output_text", None):
        full_text = full_text or response.output_text

    assert full_text.strip() or has_tool_usage, (
        "Expected either tool usage or a text response synthesizing the tool result"
    )

## QE Assertions summary

- Response status is `completed`.
- Output is present.
- Either tool_use/tool_calls appear in the output, or the final text reflects the tool result (e.g. add or greeting).